# 0. Import Libraries

In [43]:
import os, sys

def add_project_path(module_folder="model_implementation"):
    candidates = [
        os.path.abspath("."),   # current directory (Deepnote case)
        os.path.abspath("../src/")   # parent directory (local notebooks case)
    ]

    for path in candidates:
        if os.path.exists(os.path.join(path, module_folder)):
            if path not in sys.path:
                sys.path.append(path)
            return path

    raise ImportError(f"Could not find '{module_folder}' in current or parent directory")

add_project_path("model_implementation")
add_project_path("cnn")

'c:\\Users\\Aryo\\PersonalMade\\ITB Kuliah Semesteran\\Semester 6\\IF3270 Pembelajaran Mesin\\Tubes\\ML-Tubes-2_RecursiveLearnaholic\\src'

In [44]:
import subprocess
import sys
print(sys.executable)

subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow==2.21.0", "scikit-learn", "pandas", "matplotlib"])

import tensorflow as tf
print("TensorFlow successfully imported:", tf.__version__)

c:\Users\Aryo\PersonalMade\ITB Kuliah Semesteran\Semester 6\IF3270 Pembelajaran Mesin\Tubes\ML-Tubes-2_RecursiveLearnaholic\venv\Scripts\python.exe
TensorFlow successfully imported: 2.21.0


In [45]:
from pathlib import Path
from datetime import datetime

import joblib
import itertools
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

from model_implementation.ffnn import FFNN
from model_implementation.layer.loss import BCELoss
from cnn.layers import Conv2D, LocallyConnected2D, MaxPooling2D, AveragePooling2D, GlobalMaxPooling2D, GlobalAveragePooling2D, Flatten
from cnn.utility import image_loader, batch_loader, image_paths

# 1. Global Variables

In [46]:
SEED = 42
DEVICE = 'cpu'
IMAGE_SIZE = (150, 150) 
BATCH_SIZE = 32

plt.style.use("seaborn-v0_8")
np.random.seed(SEED)

CATEGORIES = {
    'buildings': 0, 
    'forest': 1,
    'glacier': 2,
    'mountain': 3,
    'sea': 4,
    'street': 5 
}
INV_CATEGORIES = {v: k for k, v in CATEGORIES.items()}

gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"GPU Detected: {gpu_devices}")
    for gpu in gpu_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
    DEVICE = "/GPU:0" 
else:
    print("No GPU found, defaulting to CPU.")
    DEVICE = "/CPU:0"

No GPU found, defaulting to CPU.


# 2. Loading Data

In [47]:
def load_intel_dataset(root_path, target_size=(150, 150)):
    """
    Loads images and labels from the directory structure:
    root/label/image.jpg
    """
    X = []
    y = []
    
    root = Path(root_path)
    for category, label in CATEGORIES.items():
        cat_path = root / category
        if not cat_path.exists():
            continue
            
        print(f"Loading {category}...")
        paths = image_paths(cat_path)
        
        for p in paths:
            try:
                img = image_loader(p, target_size=target_size)
                X.append(img)
                y.append(label)
            except Exception as e:
                print(f"Error loading {p}: {e}")
                
    return np.array(X, dtype="float32"), np.array(y, dtype="int32")

TRAIN_DIR = Path("../data/seg_train/seg_train")
TEST_DIR = Path("../data/seg_test/seg_test")
PRED_DIR = Path("../data/seg_pred/seg_pred")

print("--- Loading Training Data ---")
X_train, y_train = load_intel_dataset(TRAIN_DIR, target_size=IMAGE_SIZE)

print("\n--- Loading Test Data ---")
X_test, y_test = load_intel_dataset(TEST_DIR, target_size=IMAGE_SIZE)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=SEED)

print(f"\nFinal Shapes:")
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

--- Loading Training Data ---
Loading buildings...
Loading forest...
Loading glacier...
Loading mountain...
Loading sea...
Loading street...

--- Loading Test Data ---
Loading buildings...
Loading forest...
Loading glacier...
Loading mountain...
Loading sea...
Loading street...

Final Shapes:
Train: (11227, 150, 150, 3), Val: (2807, 150, 150, 3), Test: (3000, 150, 150, 3)


# 3. Training 16 different Model Configuration (Keras)

In [48]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import f1_score
import itertools
import pandas as pd
import numpy as np
import os
import pickle
import gc

# Setup paths relative to notebooks/
BASE_SAVE_PATH = "../models/cnn"
os.makedirs(BASE_SAVE_PATH, exist_ok=True)

# Re-loadable Generator logic to fix the 0 accuracy issue
def prepare_dataset_safe(X, y, batch_size=32, shuffle=False):
    def get_generator():
        for i in range(len(X)):
            yield X[i], y[i]
            
    dataset = tf.data.Dataset.from_generator(
        get_generator,
        output_signature=(
            tf.TensorSpec(shape=(150, 150, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int64)
        )
    )
    
    if shuffle:
        dataset = dataset.shuffle(buffer_size=500)
    
    # .repeat() ensures the generator restarts every epoch
    return dataset.batch(batch_size).repeat().prefetch(tf.data.AUTOTUNE)

# Calculate steps manually
train_steps = int(np.ceil(len(X_train) / 32))
val_steps = int(np.ceil(len(X_val) / 32))

train_ds = prepare_dataset_safe(X_train, y_train, batch_size=32, shuffle=True)
val_ds = prepare_dataset_safe(X_val, y_val, batch_size=32)
test_ds = prepare_dataset_safe(X_test, y_test, batch_size=32)

# Configurations
keras_variations = {
    'num_layers': [2, 3],        
    'filters': [32, 64],       
    'kernel_size': [3, 5],     
    'pooling_type': ['max', 'avg']  
}

keras_keys, keras_values = zip(*keras_variations.items())
keras_experiments_configs = [dict(zip(keras_keys, v)) for v in itertools.product(*keras_values)]

keras_results_registry = []

# Training Loop
for i, keras_cfg in enumerate(keras_experiments_configs):
    history_path = os.path.join(BASE_SAVE_PATH, f"history_exp_{i}.pkl")
    npy_weight_path = os.path.join(BASE_SAVE_PATH, f"keras_weights_exp_{i}.npy")
    
    if os.path.exists(history_path) and os.path.exists(npy_weight_path):
        print(f"\n>>> Skipping Experiment {i+1}/16: Already completed.")
        
        # Load already present conf
        if os.path.exists(os.path.join(BASE_SAVE_PATH, "keras_results_backup.pkl")):
            with open(os.path.join(BASE_SAVE_PATH, "keras_results_backup.pkl"), "rb") as f:
                keras_results_registry = pickle.load(f)
        continue

    print(f"\n--- Running Keras Experiment {i+1}/16: {keras_cfg} ---")

    tf.keras.backend.clear_session()
    keras_model = models.Sequential(name=f"Keras_Architecture_{i}")

    # Input Layer
    keras_model.add(layers.Conv2D(
        keras_cfg['filters'], 
        (keras_cfg['kernel_size'], keras_cfg['kernel_size']), 
        activation='relu', 
        padding='same',
        input_shape=(150, 150, 3)
    ))

    # Hidden Layers
    for _ in range(keras_cfg['num_layers'] - 1):
        keras_model.add(layers.Conv2D(
            keras_cfg['filters'], 
            (keras_cfg['kernel_size'], keras_cfg['kernel_size']), 
            activation='relu', 
            padding='same'
        ))

        if keras_cfg['pooling_type'] == 'max':
            keras_model.add(layers.MaxPooling2D((2, 2))) 
        else:
            keras_model.add(layers.AveragePooling2D((2, 2)))
            
    keras_model.add(layers.Flatten())
    keras_model.add(layers.Dense(6, activation='softmax'))

    keras_model.compile(
        optimizer='adam', 
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    # Train
    keras_history = keras_model.fit(
        train_ds, 
        epochs=10, 
        steps_per_epoch=train_steps,
        validation_data=val_ds, 
        validation_steps=val_steps,
        verbose=1
    )

    # Evaluation
    test_steps = int(np.ceil(len(X_test) / 32))
    keras_raw_preds = keras_model.predict(test_ds, steps=test_steps, verbose=0)
    keras_y_pred = np.argmax(keras_raw_preds, axis=1)
    keras_f1_macro = f1_score(y_test, keras_y_pred, average='macro')
    
    # Save Weights and Bias to .npy format
    # This creates a dictionary of layer_name: [weights, bias]
    model_weights_dict = {}
    for layer in keras_model.layers:
        weights = layer.get_weights()
        if weights:
            model_weights_dict[layer.name] = weights
            
    npy_weight_path = os.path.join(BASE_SAVE_PATH, f"keras_weights_exp_{i}.npy")
    np.save(npy_weight_path, model_weights_dict)

    history_path = os.path.join(BASE_SAVE_PATH, f"history_exp_{i}.pkl")
    with open(history_path, "wb") as f:
        pickle.dump(keras_history.history, f)

    result_entry = {
        'experiment_id': i,
        'config': keras_cfg, 
        'f1_macro': keras_f1_macro, 
        'keras_weight_path': npy_weight_path,
        'history_path': history_path
    }
    keras_results_registry.append(result_entry)

    with open(os.path.join(BASE_SAVE_PATH, "keras_results_backup.pkl"), "wb") as f:
        pickle.dump(keras_results_registry, f)
        
    del keras_model
    gc.collect()

df_keras_benchmarks = pd.DataFrame(keras_results_registry)
df_keras_benchmarks.to_csv(os.path.join(BASE_SAVE_PATH, "keras_experiment_summary.csv"), index=False)

print(f"\n[SUCCESS] All 16 Keras models, weights (.npy), and histories (.pkl) saved in: {BASE_SAVE_PATH}")


>>> Skipping Experiment 1/16: Already completed.

>>> Skipping Experiment 2/16: Already completed.

>>> Skipping Experiment 3/16: Already completed.

>>> Skipping Experiment 4/16: Already completed.

>>> Skipping Experiment 5/16: Already completed.

>>> Skipping Experiment 6/16: Already completed.

>>> Skipping Experiment 7/16: Already completed.

>>> Skipping Experiment 8/16: Already completed.

>>> Skipping Experiment 9/16: Already completed.

>>> Skipping Experiment 10/16: Already completed.

>>> Skipping Experiment 11/16: Already completed.

>>> Skipping Experiment 12/16: Already completed.

>>> Skipping Experiment 13/16: Already completed.

>>> Skipping Experiment 14/16: Already completed.

>>> Skipping Experiment 15/16: Already completed.

>>> Skipping Experiment 16/16: Already completed.

[SUCCESS] All 16 Keras models, weights (.npy), and histories (.pkl) saved in: ../models/cnn


# 4. Finding Best Configuration

In [49]:
BASE_SAVE_PATH = "../models/cnn"
final_results = []

# Re-scan for all 16 experiments
for i in range(16):
    history_file = f"history_exp_{i}.pkl"
    weight_file = f"keras_weights_exp_{i}.npy"
    
    hist_path = os.path.join(BASE_SAVE_PATH, history_file)
    weight_path = os.path.join(BASE_SAVE_PATH, weight_file)
    
    if os.path.exists(hist_path):
        with open(hist_path, "rb") as f:
            history_data = pickle.load(f)
        
        # We can reconstruct the config based on your keras_experiments_configs list
        # which should still be in your memory from the previous cells
        entry = {
            'experiment_id': i,
            'config': keras_experiments_configs[i],
            # Use the last validation accuracy as a placeholder or recalculate if needed
            'final_val_acc': history_data['val_accuracy'][-1],
            'history_path': hist_path,
            'keras_weight_path': weight_path
        }
        final_results.append(entry)

# Create a clean, synchronized summary
df_final_summary = pd.DataFrame(final_results)
df_final_summary.to_csv(os.path.join(BASE_SAVE_PATH, "keras_experiment_summary_CLEAN.csv"), index=False)

print("Summary reconstructed successfully from individual files!")
display(df_final_summary.head())

Summary reconstructed successfully from individual files!


,experiment_id,config,final_val_acc,history_path,keras_weight_path
0,0,"{'num_layers': 2, 'filters': 32, 'kernel_size'...",0.710367,../models/cnn\history_exp_0.pkl,../models/cnn\keras_weights_exp_0.npy
1,1,"{'num_layers': 2, 'filters': 32, 'kernel_size'...",0.750623,../models/cnn\history_exp_1.pkl,../models/cnn\keras_weights_exp_1.npy
2,2,"{'num_layers': 2, 'filters': 32, 'kernel_size'...",0.698254,../models/cnn\history_exp_2.pkl,../models/cnn\keras_weights_exp_2.npy
3,3,"{'num_layers': 2, 'filters': 32, 'kernel_size'...",0.686142,../models/cnn\history_exp_3.pkl,../models/cnn\keras_weights_exp_3.npy
4,4,"{'num_layers': 2, 'filters': 64, 'kernel_size'...",0.712861,../models/cnn\history_exp_4.pkl,../models/cnn\keras_weights_exp_4.npy


In [50]:
ranking_data = []

print(f"{'Rank':<5} | {'Layers':<8} | {'Filters':<8} | {'Kernel':<8} | {'Pool':<8} | {'Macro F1':<10}")
print("-" * 65)

for i, cfg in enumerate(keras_experiments_configs):
    weight_path = os.path.join(BASE_SAVE_PATH, f"keras_weights_exp_{i}.npy")
    
    if os.path.exists(weight_path):
        # Load weights into a temporary model to get predictions
        tf.keras.backend.clear_session()
        temp_model = models.Sequential()
        temp_model.add(layers.Conv2D(cfg['filters'], (cfg['kernel_size'], cfg['kernel_size']), 
                                     activation='relu', padding='same', input_shape=(150, 150, 3)))
        
        for _ in range(cfg['num_layers'] - 1):
            temp_model.add(layers.Conv2D(cfg['filters'], (cfg['kernel_size'], cfg['kernel_size']), 
                                         activation='relu', padding='same'))
            if cfg['pooling_type'] == 'max':
                temp_model.add(layers.MaxPooling2D((2, 2)))
            else:
                temp_model.add(layers.AveragePooling2D((2, 2)))
        
        temp_model.add(layers.Flatten())
        temp_model.add(layers.Dense(6, activation='softmax'))
        
        # Load the .npy weights back into the model
        weights_dict = np.load(weight_path, allow_pickle=True).item()
        trainable_layers = [l for l in temp_model.layers if len(l.get_weights()) > 0]
        for layer, (l_name, l_weights) in zip(trainable_layers, weights_dict.items()):
            layer.set_weights(l_weights)

        # Inference on test set
        test_steps = int(np.ceil(len(X_test) / 32))
        preds_raw = temp_model.predict(test_ds, steps=test_steps, verbose=0)
        y_pred = np.argmax(preds_raw, axis=1)
        
        # Calculate Macro F1
        f1 = f1_score(y_test, y_pred, average='macro')
        
        ranking_data.append({
            'exp_id': i,
            'num_layers': cfg['num_layers'],
            'filters': cfg['filters'],
            'kernel_size': cfg['kernel_size'],
            'pooling_type': cfg['pooling_type'],
            'f1_macro': f1
        })

# Sort and Print
df_ranking = pd.DataFrame(ranking_data).sort_values(by='f1_macro', ascending=False)

for rank, row in enumerate(df_ranking.itertuples(), 1):
    print(f"{rank:<5} | {row.num_layers:<8} | {row.filters:<8} | {row.kernel_size:<8} | {row.pooling_type:<8} | {row.f1_macro:.4f}")

best_config = df_ranking.iloc[0]
print(f"\n[BEST HYPERPARAMETERS]: Exp {best_config['exp_id']} with F1: {best_config['f1_macro']:.4f}")

Rank  | Layers   | Filters  | Kernel   | Pool     | Macro F1  
-----------------------------------------------------------------


c:\Users\Aryo\PersonalMade\ITB Kuliah Semesteran\Semester 6\IF3270 Pembelajaran Mesin\Tubes\ML-Tubes-2_RecursiveLearnaholic\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Aryo\PersonalMade\ITB Kuliah Semesteran\Semester 6\IF3270 Pembelajaran Mesin\Tubes\ML-Tubes-2_RecursiveLearnaholic\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Aryo\PersonalMade\ITB Kuliah Semesteran\Semester 6\IF3270 Pembelajaran Mesin\Tubes\ML-T

1     | 3        | 64       | 3        | max      | 0.7606
2     | 3        | 32       | 3        | max      | 0.7439
3     | 3        | 64       | 5        | max      | 0.7418
4     | 2        | 32       | 3        | avg      | 0.7371
5     | 3        | 64       | 3        | avg      | 0.7356
6     | 3        | 32       | 5        | avg      | 0.7340
7     | 3        | 64       | 5        | avg      | 0.7336
8     | 3        | 32       | 5        | max      | 0.7233
9     | 3        | 32       | 3        | avg      | 0.7228
10    | 2        | 64       | 3        | avg      | 0.7124
11    | 2        | 64       | 3        | max      | 0.7087
12    | 2        | 32       | 3        | max      | 0.7073
13    | 2        | 32       | 5        | max      | 0.6957
14    | 2        | 32       | 5        | avg      | 0.6890
15    | 2        | 64       | 5        | avg      | 0.6737
16    | 2        | 64       | 5        | max      | 0.6492

[BEST HYPERPARAMETERS]: Exp 12 with F1: 0.7606


# 5. Defining Best Model

In [51]:
BEST_EXP_ID = 12
best_cfg = keras_experiments_configs[BEST_EXP_ID]

tf.keras.backend.clear_session()
best_keras_model = models.Sequential(name="Champion_Model_Exp12")

best_keras_model.add(layers.Conv2D(
    best_cfg['filters'], 
    (best_cfg['kernel_size'], best_cfg['kernel_size']), 
    activation='relu', 
    padding='same',
    input_shape=(150, 150, 3)
))

for _ in range(best_cfg['num_layers'] - 1):
    best_keras_model.add(layers.Conv2D(
        best_cfg['filters'], 
        (best_cfg['kernel_size'], best_cfg['kernel_size']), 
        activation='relu', 
        padding='same'
    ))
    if best_cfg['pooling_type'] == 'max':
        best_keras_model.add(layers.MaxPooling2D((2, 2)))
    else:
        best_keras_model.add(layers.AveragePooling2D((2, 2)))

best_keras_model.add(layers.Flatten())
best_keras_model.add(layers.Dense(6, activation='softmax'))

best_weights_path = os.path.join(BASE_SAVE_PATH, f"keras_weights_exp_{BEST_EXP_ID}.npy")
best_model_weights = np.load(best_weights_path, allow_pickle=True).item()

trainable_layers = [l for l in best_keras_model.layers if len(l.get_weights()) > 0]
for layer, (l_name, l_weights) in zip(trainable_layers, best_model_weights.items()):
    layer.set_weights(l_weights)

print(f"--- Champion Model (Exp {BEST_EXP_ID}) Loaded Successfully ---")
best_keras_model.summary()

--- Champion Model (Exp 12) Loaded Successfully ---


c:\Users\Aryo\PersonalMade\ITB Kuliah Semesteran\Semester 6\IF3270 Pembelajaran Mesin\Tubes\ML-Tubes-2_RecursiveLearnaholic\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "Champion_Model_Exp12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 150, 150, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 150, 150, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 75, 75, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 75, 75, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 37, 37, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 87616)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │       525,702 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 601,350 (2.29 MB)

 Trainable params: 601,350 (2.29 MB)

 Non-trainable params: 0 (0.00 B)

# 6. Recreation with From Scratch Implementation

In [52]:
from model_implementation.layer.activation import relu, softmax
from common.autograd import Value
import numpy as np
import time

class RecursiveLearnaholicCNN:
    def __init__(self, cfg, weights_dict, shared_parameter=True):
        self.layers = []
        self.shared_parameter = shared_parameter
        self.cfg = cfg

        self.relu_act = relu()
        self.softmax_act = softmax()
        self.final_activation = self.softmax_act
        
        weight_keys = [k for k in weights_dict.keys() if 'conv2d' in k or 'dense' in k]
        current_h, current_w = 150, 150
        in_channels = 3
        
        for l_idx in range(cfg['num_layers']):
            k_w, k_b = weights_dict[weight_keys[l_idx]]
            
            if self.shared_parameter:
                pad = 1 if cfg['kernel_size'] == 3 else 2
                layer = Conv2D(in_channels, cfg['filters'], cfg['kernel_size'], padding=pad)
                layer.weight, layer.bias = k_w, k_b
            else:
                layer = LocallyConnected2D(in_channels, cfg['filters'], (current_h, current_w), cfg['kernel_size'])
                flattened_weight = k_w.reshape(-1, cfg['filters'])
                layer.weight = np.tile(flattened_weight, (current_h * current_w, 1, 1))
                layer.bias = np.tile(k_b, (current_h, current_w, 1))
            
            self.layers.append(layer)
            self.layers.append(self.relu_act)

            if l_idx < cfg['num_layers'] - 1:
                if cfg['pooling_type'] == 'max':
                    self.layers.append(MaxPooling2D())
                else:
                    self.layers.append(AveragePooling2D())

                current_h //= 2
                current_w //= 2
            
            in_channels = cfg['filters']

        self.layers.append(Flatten())
        self.dense_w, self.dense_b = weights_dict[weight_keys[-1]]

    def forward(self, x):
        out = x
        for layer in self.layers:
            if hasattr(out, 'data'): 
                out = out.data
                
            out = layer.forward(out)
        
        if hasattr(out, 'data'): 
            out = out.data
            
        logits = (out @ self.dense_w) + self.dense_b
        return self.final_activation.forward(logits)

    def count_params(self):
        total = 0
        for layer in self.layers:
            if hasattr(layer, 'weight') and layer.weight is not None:
                total += layer.weight.size + layer.bias.size
        total += self.dense_w.size + self.dense_b.size
        return total

In [53]:
model_shared = RecursiveLearnaholicCNN(best_cfg, best_model_weights, shared_parameter=True)
model_non_shared = RecursiveLearnaholicCNN(best_cfg, best_model_weights, shared_parameter=False)

p_shared = model_shared.count_params()
p_non_shared = model_non_shared.count_params()

print(f"--- Parameters between Shared and Non-Shared ---")
print(f"Shared Parameters:     {p_shared:,}")
print(f"Non-Shared Parameters: {p_non_shared:,}")
print(f"Parameter Explosion:   {p_non_shared / p_shared:.2f}x")

--- Parameters between Shared and Non-Shared ---
Shared Parameters:     601,350
Non-Shared Parameters: 299,120,134
Parameter Explosion:   497.41x


In [55]:
num_samples = 100
X_test_sample = X_test[:num_samples]
y_test_sample = y_test[:num_samples]


k_out_l1 = best_keras_model.layers[0](X_test_sample).numpy()
s_out_l1 = model_shared.layers[0].forward(X_test_sample)

print(f"L1 (Conv2D) Match: {np.allclose(k_out_l1, s_out_l1, atol=1e-5)}")
if not np.allclose(k_out_l1, s_out_l1, atol=1e-5):
    print(f"Mean Absolute Error: {np.mean(np.abs(k_out_l1 - s_out_l1))}")
    print(f"Keras Shape: {k_out_l1.shape} | Scratch Shape: {s_out_l1.shape}")

k_out_l2 = best_keras_model.layers[1](k_out_l1).numpy()
s_out_l2 = model_shared.layers[1].forward(s_out_l1)
if hasattr(s_out_l2, 'data'): s_out_l2 = s_out_l2.data

print(f"L2 (ReLU) Match:   {np.allclose(k_out_l2, s_out_l2, atol=1e-5)}")

L1 (Conv2D) Match: False
Mean Absolute Error: 0.10975092995691547
Keras Shape: (100, 150, 150, 64) | Scratch Shape: (100, 150, 150, 64)
L2 (ReLU) Match:   False


In [ ]:
k_kernels = best_keras_model.layers[0].get_weights()[0]
s_kernels = model_shared.layers[0].weight

print(f"Kernel Match: {np.allclose(k_kernels, s_kernels)}")
# If this is False, your weight loading loop is assigning the wrong array to the wrong layer.

Kernel Match: True


In [57]:
# Extract Keras output BEFORE ReLU if possible, or apply ReLU to scratch
k_out_l1 = best_keras_model.layers[0](X_test_sample).numpy()

# Apply ReLU to scratch to match Keras's fused activation
s_out_l1_raw = model_shared.layers[0].forward(X_test_sample)
s_out_l1 = np.maximum(0, s_out_l1_raw)

print(f"L1 (Conv2D+ReLU) Match: {np.allclose(k_out_l1, s_out_l1, atol=1e-3)}")

L1 (Conv2D+ReLU) Match: True


# 7. Comparison

## 7.1. Shared Parameter From-Scratch vs Keras

In [ ]:
import time

# Only use 100 because Numpy is slower than keras
num_samples = 100
X_test_sample = X_test[:num_samples]
y_test_sample = y_test[:num_samples]

# --- Keras Prediction ---
start_k = time.time()
keras_raw = best_keras_model.predict(X_test_sample, verbose=0)
keras_preds = np.argmax(keras_raw, axis=1)
keras_time = time.time() - start_k
f1_keras = f1_score(y_test_sample, keras_preds, average='macro')

# --- Scratch Shared Prediction ---
start_s = time.time()
scratch_shared_raw = model_shared.forward(X_test_sample)
# Ensure scratch_shared_raw is data array if it returns a Value object
if hasattr(scratch_shared_raw, 'data'): scratch_shared_raw = scratch_shared_raw.data
scratch_shared_preds = np.argmax(scratch_shared_raw, axis=1)
scratch_shared_time = time.time() - start_s
f1_scratch_shared = f1_score(y_test_sample, scratch_shared_preds, average='macro')

print(f"--- Shared Scratch vs Keras (n={num_samples}) ---")
print(f"Keras Macro F1:          {f1_keras:.5f}")
print(f"Scratch Shared Macro F1: {f1_scratch_shared:.5f}")
print(f"Execution Time (Keras):  {keras_time:.4f}s")
print(f"Execution Time (Scratch): {scratch_shared_time:.4f}s")
print(f"Numerical Match:         {np.allclose(keras_raw, scratch_shared_raw, atol=1e-5)}")

--- Shared Scratch vs Keras (n=100) ---
Keras Macro F1:          0.14394
Scratch Shared Macro F1: 0.08271
Execution Time (Keras):  0.6183s
Execution Time (Scratch): 6.0021s
Numerical Match:         False


## 7.2. Shared Parameter From-Scratch vs Non-Shared Parameter From-Scratch

In [ ]:
# --- Non-Shared Execution ---
# Only 5 because slow
small_batch = X_test[:5]
small_y = y_test[:5]

scratch_ns_raw = model_non_shared.forward(small_batch)
if hasattr(scratch_ns_raw, 'data'): scratch_ns_raw = scratch_ns_raw.data
scratch_ns_preds = np.argmax(scratch_ns_raw, axis=1)
f1_ns = f1_score(small_y, scratch_ns_preds, average='macro')

print(f"\n--- Shared vs Non-Shared Comparison ---")
print(f"Shared Params:           {p_shared:,}")
print(f"Non-Shared Params:       {p_non_shared:,}")
print(f"Explosion Factor:        {p_non_shared / p_shared:.2f}x")
print(f"Non-Shared Macro F1 (small batch): {f1_ns:.5f}")